In [ ]:
import librosa

audio = 'audio.wav'
user_audio, sr = librosa.load(audio, sr=22050)
print(f"Loaded! Duration: {len(user_audio)/sr:.2f}s")

In [ ]:
import numpy as np

def trim_silence(audio, sr, threshold_db=-40):
    """Remove silence from beginning and end of audio"""

    # Create spectrogram and convert to dB
    S = librosa.feature.melspectrogram(y=audio, sr=sr)
    S_db = librosa.power_to_db(S, ref=np.max)

    # Get energy per frame
    energy = np.mean(S_db, axis=0)

    # Find frames above threshold (-40 dB soft singing)
    active_frames = energy > threshold_db
    active_indices = np.where(active_frames)[0]

    if len(active_indices) == 0:
        return audio

    # Get start and end frames
    start_frame = active_indices[0]
    end_frame = active_indices[-1]

    # Convert to samples
    start_sample = librosa.frames_to_samples(start_frame, sr=sr)
    end_sample = librosa.frames_to_samples(end_frame, sr=sr)

    # Add 100ms buffer for smooth transitions
    buffer = int(0.1 * sr)
    start_sample = max(0, start_sample - buffer)
    end_sample = min(len(audio), end_sample + buffer)

    # Trim and return
    audio_trimmed = audio[start_sample:end_sample]

    print(f"✓ Trimmed silence!")
    print(f"  Original: {len(audio)/sr:.2f}s")
    print(f"  Trimmed:  {len(audio_trimmed)/sr:.2f}s")
    print(f"  Removed:  {(len(audio) - len(audio_trimmed))/sr:.2f}s of silence")

    return audio_trimmed

In [ ]:
## trimming audio through function
audio_trimmed = trim_silence(audio, sr, threshold_db=-40)

# Save trimmed audio
import soundfile as sf
sf.write('audio_trimmed.wav', audio_trimmed, sr)
print("Saved trimmed audio to 'audio_trimmed.wav'")